<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. DataFrame Relational Operations</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 3.2 Demo - DataFrame Relational Operations in Spark
### Duration - 30 Minutes

This demonstration combines the Bakehouse sample tables - transactions, customers, franchises, and suppliers - using joins and set operations, and shows how to verify the join strategy Spark chose.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Join DataFrames on common and differently named key columns
- Use full outer joins to find orphaned records across datasets
- Express joins in Spark SQL over temporary views
- Apply set operations to compare DataFrames
- Request and verify a broadcast join

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run, sets your default catalog, and sets the default schema to `spark_dev`.

In [0]:
%run ../Includes/Classroom-Setup-3

## B. Setup and Data Loading

The Bakehouse dataset models a cookie franchise business: customers buy products at franchises, and franchises buy ingredients from suppliers.

In [0]:
%python
# Read the Bakehouse Tables
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")
customers_df = spark.read.table("samples.bakehouse.sales_customers")
franchises_df = spark.read.table("samples.bakehouse.sales_franchises")
suppliers_df = spark.read.table("samples.bakehouse.sales_suppliers")

In [0]:
%python
# Examine the Table Schemas
print("Transactions:")
transactions_df.printSchema()
print("Customers:")
customers_df.printSchema()
print("Franchises:")
franchises_df.printSchema()
print("Suppliers:")
suppliers_df.printSchema()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Note the key columns: <code>transactions</code> carries both <code>customerID</code> and <code>franchiseID</code>, <code>franchises</code> carries <code>supplierID</code>. These shared keys are what we join on.</p>
</div>

## C. Basic Join Operations

When the key column has the same name on both sides, pass the column name as the join condition - Spark keeps a single copy of the key column in the result.

In [0]:
%python
# Join Transactions to Franchises
# Inner join on the common franchiseID column
sales_with_franchise_df = transactions_df.join(
    franchises_df,
    on="franchiseID",
    how="inner"
)

display(sales_with_franchise_df.limit(10))

In [0]:
%python
# Project Before Joining
from pyspark.sql.functions import col

# Select only the columns each side contributes before joining
# (less data to shuffle, no ambiguous column names)
franchise_locations_df = franchises_df.select(
    "franchiseID",
    col("name").alias("franchise_name"),
    col("city").alias("franchise_city"),
    col("country").alias("franchise_country")
)

sales_by_location_df = (
    transactions_df
    .select("transactionID", "franchiseID", "product", "totalPrice")
    .join(franchise_locations_df, on="franchiseID", how="inner")
)

display(sales_by_location_df.limit(10))

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Both tables carry generic column names like <code>name</code> and <code>city</code>. Renaming with <code>alias()</code> during projection avoids ambiguous references after the join and shuffles less data - projection before a join is a cheap, reliable optimization.</p>
</div>

In [0]:
%python
# Aggregate the Joined Result
from pyspark.sql.functions import sum, round, desc

# Which cities generate the most revenue?
display(
    sales_by_location_df
    .groupBy("franchise_city", "franchise_country")
    .agg(round(sum("totalPrice"), 2).alias("total_revenue"))
    .orderBy(desc("total_revenue"))
    .limit(10)
)

### Joining with an Expression

When key columns have different names (or you need multiple conditions), pass a join expression. Here we relate franchises to suppliers - and rename the conflicting columns first.

In [0]:
%python
# Join Franchises to Suppliers with an Expression
# Rename overlapping column names before the join
suppliers_renamed_df = (
    suppliers_df
    .withColumnRenamed("name", "supplier_name")
    .withColumnRenamed("city", "supplier_city")
    .withColumnRenamed("size", "supplier_size")
)

# Join using an explicit column equality expression
franchise_suppliers_df = franchises_df.join(
    suppliers_renamed_df,
    on=franchises_df.supplierID == suppliers_renamed_df.supplierID,
    how="inner"
)

display(
    franchise_suppliers_df
    .select("name", "city", "supplier_name", "supplier_city", "ingredient")
    .limit(10)
)

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">With an expression join, <strong>both</strong> <code>supplierID</code> columns survive into the result (unlike the common-column form, which keeps one). Drop one of them, or project the columns you need, before saving the result.</p>
</div>

## D. Full Outer Join Operations

Outer joins reveal what does *not* match. Let's find franchises whose supplier is missing from the suppliers table, and suppliers no franchise uses.

In [0]:
%python
# Full Outer Join Franchises and Suppliers
from pyspark.sql.functions import col

franchise_supplier_audit_df = (
    franchises_df
    .select("franchiseID", col("name").alias("franchise_name"), "supplierID")
    .join(
        suppliers_renamed_df.select("supplierID", "supplier_name"),
        on=franchises_df.supplierID == suppliers_renamed_df.supplierID,
        how="full_outer"
    )
)

# Keep only the rows where one side failed to match
orphaned_records_df = franchise_supplier_audit_df.filter(
    col("franchise_name").isNull() | col("supplier_name").isNull()
)

display(orphaned_records_df)

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Rows with a null <code>supplier_name</code> are franchises pointing at a supplier that does not exist; rows with a null <code>franchise_name</code> are suppliers no franchise buys from. This referential integrity audit is a classic full outer join use case.</p>
</div>

## E. Joins in Spark SQL

The same joins can be expressed in SQL over temporary views - the engine and the resulting plans are identical.

In [0]:
%python
# Register Temp Views
franchises_df.createOrReplaceTempView("franchises")
suppliers_df.createOrReplaceTempView("suppliers")

In [0]:
-- Full Outer Join in SQL
SELECT
  f.franchiseID,
  f.name AS franchise_name,
  s.supplierID,
  s.name AS supplier_name
FROM franchises f
FULL OUTER JOIN suppliers s
  ON f.supplierID = s.supplierID
WHERE f.name IS NULL OR s.name IS NULL;

## F. Set Operations

Set operations compare DataFrames row by row. Both inputs must have matching schemas, so we compare single-column projections: the cities where customers live versus the cities where franchises operate.

In [0]:
%python
# Cities with Both Customers and Franchises
customer_cities_df = customers_df.select("city").distinct()
franchise_cities_df = franchises_df.select("city").distinct()

# Intersection: cities that appear in both DataFrames
served_cities_df = customer_cities_df.intersect(franchise_cities_df)

display(served_cities_df)

In [0]:
%python
# Customer Cities with No Franchise
# Subtraction: customer cities that have no franchise
unserved_cities_df = customer_cities_df.subtract(franchise_cities_df)

print(f"{unserved_cities_df.count()} customer cities have no franchise")
display(unserved_cities_df.limit(10))

In [0]:
%python
# Union of All Known Cities
# Union keeps duplicates (like SQL UNION ALL), so deduplicate explicitly
all_cities_df = customer_cities_df.union(franchise_cities_df).distinct()

print(f"Customer cities: {customer_cities_df.count()}")
print(f"Franchise cities: {franchise_cities_df.count()}")
print(f"All cities (union, deduplicated): {all_cities_df.count()}")

## G. Broadcast Joins

The franchises table is tiny compared to transactions, making it an ideal broadcast candidate: Spark copies it to every Executor and avoids shuffling the transaction data.

In [0]:
%python
# Request a Broadcast Join
from pyspark.sql.functions import broadcast

broadcast_join_df = transactions_df.join(
    broadcast(franchises_df),
    on="franchiseID",
    how="inner"
)

# Inspect the physical plan: look for BroadcastHashJoin
broadcast_join_df.explain()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The physical plan shows <code>BroadcastHashJoin</code> instead of <code>SortMergeJoin</code> - the small table is shipped to every Executor and no shuffle of the large side occurs. Spark broadcasts small tables automatically below the <code>spark.sql.autoBroadcastJoinThreshold</code>; the hint makes the choice explicit.</p>
</div>

## Key Takeaways

**Join strategy:**

- Use the common-column form (`on="key"`) when names match; use expressions when they differ, and rename conflicting columns first
- Outer joins plus `isNull()` filters are the standard pattern for finding orphaned records

**Set operations:**

- `intersect()`, `subtract()`, and `union()` compare whole rows and require matching schemas; remember `union()` keeps duplicates

**Performance:**

- Project only needed columns before joining
- Broadcast small tables, and verify the chosen strategy with `explain()` or the Spark UI

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>